In [ ]:
from models.models import train_model_on_named_experiment

## Training Models:

#### Implemented Models:
| Model Class | Model Description | 
|------------|--------------------|  
| `AE` | Autoencoder | 
| `AEC`| Autoencoder Classifier |
| `scMEDAL-FE` | Domain Adversarial Autoencoder |
| `scMEDAL-FEC`| Domain Adversarial Autoencoder Classifier |
| `scMEDAL-RE`| Domain Enhancing Autoencoder Classifier | 

#### Named Experiments:
| Valid Named Experiment | Dataset |  n_clusters | n_pred |
|------------------------|---------|-------------|--------|
| `AML`| Acute Myeloid Leukemia | 19 | 21 |
| `ASD`| Autism Spectrum Disorder | 31 | 17 | 
| `HH` | Healthy Heart | 147 | 13 | 

**Note:** If training on other datasets, the configs will need to be passed in as dictionaries to `model_kwargs` and `train_kwargs`.

`quick` is a boolean flag that can be passed to `train_kwargs` which shortens training to only 1 fold for 3 epochs.

In [ ]:
ae_aml = train_model_on_named_experiment("AE", "AML", train_kwargs={"quick":True})

In [ ]:
scmedalre_asd = train_model_on_named_experiment("scMEDAL-RE", "ASD", model_kwargs={"n_latent_dims":50}, train_kwargs={"quick":True})

## Analyzing Models:

In [1]:
import os
from importlib import reload
import analysis.analysis as aa
reload(aa)
import analysis.analysis as aa

experiment_id = "AML_default"

latent_space_header = os.path.join(os.getcwd(),"outputs/AML/latent_space/log_transformed_2916hvggenes")
model_saved_header = os.path.join(os.getcwd(),"outputs/AML/saved_models/log_transformed_2916hvggenes")

model_folder_dict = {
    "ae":"run_crossval_n_latent_dims-2_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-ae_batch_size-512_epochs-500_patience-30_compute_latents_callback-False_sample_size-10000_n_components-2_2025-06-25_09-45",
    "aec":"run_crossval_n_latent_dims-2_layer_units-512-132_n_pred-21_use_batch_norm-True_scaling-min_max_model_type-aec_batch_size-512_epochs-500_patience-30_compute_latents_callback-False_sample_size-10000_2025-06-25_09-45",
    "scmedalfe":"run_crossval_loss_gen_weight-1_loss_recon_weight-4000_loss_class_weight-1_n_latent_dims-2_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfe_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-06-25_09-45",
    "scmedalfec":"run_crossval_loss_gen_weight-1_loss_recon_weight-2000_loss_class_weight-1_n_latent_dims-2_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfec_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-06-25_09-45",
    "scmedalre":"run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-2_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-06-25_10-37",
}
latent_space_paths = {k:os.path.join(latent_space_header, k) for k in model_folder_dict.keys() }
model_paths = {k:os.path.join(model_saved_header, k) for k in model_folder_dict.keys() }

aml_analysis = aa.AMLAnalysis(saved_model_paths=model_paths, latent_space_paths=latent_space_paths)


2025-06-25 16:19:57.078042: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-25 16:19:57.143585: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-25 16:19:57.144741: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-25 16:19:59.433584: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [ ]:
aml_analysis.clustering_scores(model_folder_dict, experimentid=experiment_id)

In [2]:
df = aml_analysis.genomap(model_folder_dict, experimentid=experiment_id)

Index(['Key', 'Split', 'ReconPath', 'Type'], dtype='object') Index(['Split', 'Type', 'InputsPath'], dtype='object')
splits path: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/data/AML_data/log_transformed_2916hvggenes/splits
Reading paths,
df paths:   Key  Split                                          ReconPath   Type  \
0  ae      1  /endosome/archive/bioinformatics/DLLab/src/Aus...  train   
1  ae      1  /endosome/archive/bioinformatics/DLLab/src/Aus...    val   
2  ae      1  /endosome/archive/bioinformatics/DLLab/src/Aus...   test   
3  ae      2  /endosome/archive/bioinformatics/DLLab/src/Aus...  train   
4  ae      2  /endosome/archive/bioinformatics/DLLab/src/Aus...    val   

                                          InputsPath recon_prefix  
0  /archive/bioinformatics/DLLab/AixaAndrade/src/...  recon_train  
1  /archive/bioinformatics/DLLab/AixaAndrade/src/...    recon_val  
2                                                NaN   reco

/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


adata_multibatch_n_cells.X (300, 2916)
adata_multibatch_n_cells.obs      Unnamed: 0.2  Unnamed: 0  Unnamed: 0.1                      Cell  \
0           13066       13940         13940  AML707B-D97_CGAATGCCAGCN   
1            4493        5205          5205  AML707B-D41_CTGCTGCTTTCA   
2           13509       14383         14383   AML1012-D0_CTGATGACGTGT   
3           13190       14064         14064   AML1012-D0_ACGCTTTATGCA   
4           33966       35699         35699   BM5-34p38n_TGTGGACAAGCA   
..            ...         ...           ...                       ...   
295         12650       13362         13362    AML556-D0_GACGCGCGAGTT   
296         16414       17288         17288   AML419A-D0_CCTGGGACTCCN   
297         16214       17088         17088   AML419A-D0_TCGCGTGTAGCC   
298         15847       16721         16721   AML419A-D0_GCTAAGATGCTN   
299         14208       15082         15082   AML1012-D0_AGCAACGGTCAT   

     NumberOfReads  AlignedToGenome  AlignedToTranscrip

/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/ot/lp/__init__.py:388: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)
/endosome/archive/bioinformatics/DLLab/src/AustinMarckx/git/scMEDAL_for_scRNAseq/utils/genomaps_utils.py:261: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.1

Missing indices: set()
genomap stored in /endosome/archive/bioinformatics/DLLab/src/AustinMarckx/git/scMEDAL_for_scRNAseq/outputs/AML/compare_models/AML_default/300cells_per_batch_19batches_train_1_Mono_Mono-like
genomap_path /endosome/archive/bioinformatics/DLLab/src/AustinMarckx/git/scMEDAL_for_scRNAseq/outputs/AML/compare_models/AML_default/300cells_per_batch_19batches_train_1_Mono_Mono-like/genomap_300cells_per_batch_19batches_train_1_Mono_Mono-like.npy
cm_multibatch_path /endosome/archive/bioinformatics/DLLab/src/AustinMarckx/git/scMEDAL_for_scRNAseq/outputs/AML/compare_models/AML_default/CMmultibatch_300_cells_per_batch_19batches_Mono_Mono-like
obs multibatch      Unnamed: 0.2  Unnamed: 0  Unnamed: 0.1                      Cell  \
0           13066       13940         13940  AML707B-D97_CGAATGCCAGCN   
1            4493        5205          5205  AML707B-D41_CTGCTGCTTTCA   
2           13509       14383         14383   AML1012-D0_CTGATGACGTGT   
3           13190       14064     

In [ ]:
cell_id = "AML707B-D41_TATCGGATCCCT"
df
print(df.columns)

original_batch = df.loc[(df["Cell"] == cell_id) & (df["recon_prefix"].str.contains("batch"))]#.values[0]
original_batch

Index(['Unnamed: 0.2', 'Unnamed: 0', 'Unnamed: 0.1', 'Cell', 'NumberOfReads',
       'AlignedToGenome', 'AlignedToTranscriptome', 'TranscriptomeUMIs',
       'NumberOfGenes', 'CyclingScore', 'CyclingBinary', 'MutTranscripts',
       'WtTranscripts', 'PredictionRF2', 'PredictionRefined', 'CellType',
       'Score_HSC', 'Score_Prog', 'Score_GMP', 'Score_ProMono', 'Score_Mono',
       'Score_cDC', 'Score_pDC', 'Score_earlyEry', 'Score_lateEry',
       'Score_ProB', 'Score_B', 'Score_Plasma', 'Score_T', 'Score_CTL',
       'Score_NK', 'NanoporeTranscripts', 'id', 'Day', 'unique_id',
       'Patient_group', 'celltype', 'batch', 'n_genes', 'original_index',
       'recon_prefix'],
      dtype='object')


IndexError: index 0 is out of bounds for axis 0 with size 0